# Notebook 3 — Interactive Mapping with folium

**Goal:** go from Notebook 1's one-liner `.explore()` map to a fully custom,
multi-layer map — styled footprints, colored infrastructure points, a legend — built
directly with `folium`.

`/risk/features/multi.geojson` is the workhorse endpoint here: give it a dam number
and (optionally) which targets you want, or `all=true` for everything, and it
returns every intersecting feature — points, lines, and polygons — in a single
GeoJSON `FeatureCollection`, tagged with a `target` property so you know what each
feature is.


In [ ]:
import requests
import pandas as pd
import geopandas as gpd
import folium

BASE_URL = "http://149.165.154.170:30080"
DAM = "UT00221"  # Mountain Dell

features_json = requests.get(
    f"{BASE_URL}/risk/features/multi.geojson", params={"damnumber": DAM, "all": "true"}, timeout=20
).json()

features_gdf = gpd.GeoDataFrame.from_features(features_json["features"], crs="EPSG:4326")
features_gdf["target"].value_counts()


## Building the map layer by layer

We'll style three groups differently, since they need different treatment:
- **The inundation zone** — one polygon outline, no fill, so it doesn't hide everything else.
- **Point infrastructure** (hospitals, power plants, aviation, hazardous waste, wwtp) — colored circle markers with popups.
- **Lines and polygons** (railroads, transportation, gap_status, svi_tracts) — colored by type with a legend.


In [ ]:
zone_json = requests.get(f"{BASE_URL}/risk/zone.geojson", params={"damnumber": DAM}, timeout=10).json()
zone_gdf = gpd.GeoDataFrame.from_features(zone_json["features"], crs="EPSG:4326")

center = zone_gdf.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=11, tiles="cartodbpositron")

folium.GeoJson(
    zone_gdf,
    name="Inundation zone",
    style_function=lambda f: {"color": "crimson", "weight": 2, "fillOpacity": 0},
).add_to(m)

m


In [ ]:
POINT_STYLE = {
    "hospitals": {"color": "red", "icon": "plus-sign"},
    "power_plants": {"color": "orange", "icon": "flash"},
    "aviation": {"color": "blue", "icon": "plane"},
    "hazardous_waste": {"color": "black", "icon": "warning-sign"},
    "wwtp": {"color": "darkgreen", "icon": "tint"},
    "dams": {"color": "purple", "icon": "home"},
}

point_targets = [t for t in POINT_STYLE if t in features_gdf["target"].unique()]
points_gdf = features_gdf[features_gdf["target"].isin(point_targets)]

for _, row in points_gdf.iterrows():
    style = POINT_STYLE[row["target"]]
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        tooltip=row["target"],
        icon=folium.Icon(color=style["color"], icon=style["icon"], prefix="glyphicon"),
    ).add_to(m)

m


In [ ]:
LINE_COLORS = {
    "railroads": "gray",
    "transportation": "steelblue",
    "ng_pipelines": "goldenrod",
    "transmission": "darkred",
}

for target, color in LINE_COLORS.items():
    subset = features_gdf[features_gdf["target"] == target]
    if subset.empty:
        continue
    folium.GeoJson(
        subset,
        name=target,
        style_function=lambda f, color=color: {"color": color, "weight": 2},
        tooltip=folium.GeoJsonTooltip(fields=["target"]),
    ).add_to(m)

m


In [ ]:
POLY_COLORS = {"gap_status": "forestgreen", "svi_tracts": "purple"}

for target, color in POLY_COLORS.items():
    subset = features_gdf[features_gdf["target"] == target]
    if subset.empty:
        continue
    folium.GeoJson(
        subset,
        name=target,
        style_function=lambda f, color=color: {"color": color, "fillColor": color, "weight": 1, "fillOpacity": 0.25},
        tooltip=folium.GeoJsonTooltip(fields=["target"]),
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m


## One implementation among many

Everything above was *our* implementation choice — our basemap, our icon set, our
colors, our own custom folium layers built cell by cell. There's nothing uniquely
correct about it; it's just one way to visualize the same underlying zone and
infrastructure data this API serves.

This project also ships a public reference visualization built on the exact same
backend: the [dam risk dashboard](https://dams.i-guide.io/). It has its own
per-dam map view — same dam, same PostGIS tables and REST API underneath, but a
completely different frontend (its own React/Leaflet app, not folium), a
different basemap, a different icon/legend design, aimed at a different audience
(dam safety officers and the public, not developers exploring an API). Since it's
just a webpage, we can embed it live, right here, using the same `DAM` variable —
change `DAM` above and this updates too.


In [ ]:
from IPython.display import IFrame

IFrame(f"https://dams.i-guide.io/dam-map/{DAM}", width=950, height=650)


Compare it to the map you just built: different basemap tiles, a fixed on-map
legend instead of folium's toggleable `LayerControl`, its own icon set for
infrastructure points. Same data, same API, two entirely different — and equally
valid — visualization choices. The "right" one depends on who's looking at it.


## Bonus: comparing two dams at once

The dashboard above only ever shows **one** dam per page — that's a limitation of
*its* implementation, not of the underlying data. Since we're just making our own
HTTP requests, nothing stops us from pulling two dams and putting them on the same
map. **Little Dell (`UT00755`)** sits about 10 miles up the same canyon from
Mountain Dell — close enough to compare side by side, and it's the dam suggested in
the exercise below anyway.

Two more API calls, tagged with which dam they came from:


In [ ]:
DAM_2 = "UT00755"  # Little Dell -- ~10 miles up the canyon from Mountain Dell

zone_gdf_2 = gpd.GeoDataFrame.from_features(
    requests.get(f"{BASE_URL}/risk/zone.geojson", params={"damnumber": DAM_2}, timeout=10).json()["features"],
    crs="EPSG:4326",
)
features_gdf_2 = gpd.GeoDataFrame.from_features(
    requests.get(f"{BASE_URL}/risk/features/multi.geojson", params={"damnumber": DAM_2, "all": "true"}, timeout=20).json()["features"],
    crs="EPSG:4326",
)

# /risk/zone.geojson already tags each zone with its dam -- multi.geojson's other
# targets don't, so we tag those two GeoDataFrames ourselves before combining them.
features_gdf["damnumber"] = DAM
features_gdf_2["damnumber"] = DAM_2

zone_gdf_2[["dam_name", "damnumber"]]


Two zones first, styled by dam instead of by target, so you can tell them apart —
then `m2.fit_bounds(...)` so folium zooms to fit both automatically instead of us
guessing a zoom level:


In [ ]:
ZONE_COLORS = {DAM: "crimson", DAM_2: "royalblue"}

m2 = folium.Map(tiles="cartodbpositron")

for dam, zgdf in [(DAM, zone_gdf), (DAM_2, zone_gdf_2)]:
    folium.GeoJson(
        zgdf,
        name=f"{zgdf['dam_name'].iloc[0]} zone",
        style_function=lambda f, color=ZONE_COLORS[dam]: {
            "color": color, "weight": 3, "fillColor": color, "fillOpacity": 0.05
        },
        tooltip=folium.GeoJsonTooltip(fields=["dam_name", "damnumber"]),
    ).add_to(m2)

m2.fit_bounds(m2.get_bounds())
m2


Now the same `POLY_COLORS` styling as the cell above — except each target's layer
is built from **both** dams' matching rows combined with `pd.concat`, so a single
checkbox in the layer control lights up matching features near *either* zone at once:


In [ ]:
POLY_COLORS = {"gap_status": "forestgreen", "svi_tracts": "purple"}

for target, color in POLY_COLORS.items():
    combined = pd.concat(
        [features_gdf[features_gdf["target"] == target], features_gdf_2[features_gdf_2["target"] == target]],
        ignore_index=True,
    )
    if combined.empty:
        continue
    folium.GeoJson(
        combined,
        name=target,
        style_function=lambda f, color=color: {"color": color, "fillColor": color, "weight": 1, "fillOpacity": 0.25},
        tooltip=folium.GeoJsonTooltip(fields=["target", "damnumber"]),
    ).add_to(m2)

folium.LayerControl(collapsed=False).add_to(m2)
m2


Zoom out and both zones are visible on the same map — Mountain Dell outlined in
crimson, Little Dell in royal blue. Toggle `gap_status` or `svi_tracts` in the layer
control and matching features light up near *both* zones at once, not just one.
The dashboard can't do that today; it's built to render one dam per page. Nothing
about the API forced that limitation on it — two `requests.get()` calls and a
`pd.concat()` was the whole trick.


## Exercise

Pick a different dam (try `"UT00755"`, Little Dell — it has the highest interstate
mileage impact in the dataset) and re-run this notebook top to bottom. Then try
adding a new layer for a target we didn't style above — `interstates_impact` or
`ushighway_impact` are good candidates (check `features_gdf["target"].unique()`
first to confirm they're present for your chosen dam).
